# Plot Cell Entry Effects of RSV Sequence Variation

This notebook analyzes the cell entry functional effects of naturally occurring RSV F protein sequence variation identified from sequence alignments.

## Reference Strains

The following reference strains are used for identifying differences:
- **RSV-A**: hRSV/A/England/397/2017 (GenBank: PP109421.1, Protein: WTM05212.1)
- **RSV-B**: HRSV/B/AUS/VIC-RCH056/2019 (GenBank: OP975389.1, Protein: WBQ20026.1)
- **DMS Long strain**: RSV_Long_F (the strain used for deep mutational scanning experiments)

**Terminology**:
- **Difference from reference strain**: A difference between a sequence and the RSV-A or RSV-B reference strain
- **Difference from DMS strain**: A difference between a sequence and the RSV_Long_F strain used in DMS experiments
- **Sequence variation**: General term for any amino acid differences between sequences

## Cell Entry Effect Calculation

This notebook supports three types of analyses:

### Analysis 1: Differences from DMS Long Strain
- **Identifies differences** relative to DMS Long strain
- **Calculates effects** relative to DMS Long strain
- Example: DMS has N at position 100, sequence has K
  - Effect = effect(N→K) - effect(N→N) = effect(N→K) (raw DMS effect)

### Analysis 2: Differences from Reference Strain
- **Identifies differences** relative to RSV-A/B reference strain
- **Calculates effects** relative to RSV-A/B reference strain
- Example: RSV-A ref has K at position 100, sequence has L, DMS has N
  - Effect = effect(N→L) - effect(N→K) (differential effect)

### Analysis 3: DMS Differences with Reference-Relative Effects
- **Identifies ALL differences** relative to DMS Long strain (includes everything)
- **Calculates effects intelligently** based on whether sequence matches the RSV-A/B reference:

**Case 1: Sequence matches reference** (e.g., position 101: DMS=T, Ref=P, Seq=P)
  - Use raw DMS effect: effect(T→P) - effect(T→T) = effect(T→P)
  - Example: -2.729

**Case 2: All three amino acids differ** (e.g., DMS=N, Ref=K, Seq=L)
  - Use differential effect: effect(N→L) - effect(N→K)
  - This shows the effect of the sequence variant relative to the natural reference

This approach implements the SequenceScorer logic: for any difference from amino acid A to B at a site, calculate effect(DMS_wt→B) - effect(DMS_wt→A), where A and B can be any amino acids measured in the DMS.

In [ ]:
# Parameters - will be overridden by papermill
strain = "RSV-A"
variations_with_effects_file = "results/sequence_variation/RSV-A_sequence_variations_with_effects.csv"
output_html = "results/sequence_variation/RSV-A_sequence_variation_effects.html"
analysis_description = "Differences from DMS Long Strain"
mutation_identified_relative_to = "RSV_Long_F"
effects_calculated_relative_to = "RSV_Long_F"

In [ ]:
import pandas as pd
import altair as alt
from Bio import SeqIO

_ = alt.data_transformers.disable_max_rows()

## Configuration

In [ ]:
print(f"Analyzing strain: {strain}")
print(f"Analysis type: {analysis_description}")
print(f"Variations with effects: {variations_with_effects_file}")
print(f"Differences identified relative to: {mutation_identified_relative_to}")
print(f"Effects calculated relative to: {effects_calculated_relative_to}")
print(f"Output HTML: {output_html}")

## Load data

In [ ]:
# Load sequence variations with effects
with_effects = pd.read_csv(variations_with_effects_file)
print(f"Loaded {len(with_effects)} {strain} sequence variations")
print(f"At {with_effects['site'].nunique()} sites")

# Separate those with and without DMS data
with_dms = with_effects.dropna(subset=['cell entry'])
without_dms = with_effects[with_effects['cell entry'].isna()]

print(f"\nVariations with DMS data: {len(with_dms)}")
print(f"Variations without DMS data: {len(without_dms)}")
print(f"Coverage: {len(with_dms) / len(with_effects) * 100:.1f}% of variations have DMS data")

display(with_effects.head())

# Also load all possible DMS mutations for comparison in plots
# Extract cell_entry_file path from the pipeline
import os
cell_entry_file = "results/summaries/cell_entry.csv"
cell_entry = pd.read_csv(cell_entry_file)
print(f"\nLoaded {len(cell_entry)} total possible mutations from DMS data for comparison")

## Summary statistics

In [ ]:
print(f"Cell Entry Effect Statistics for {strain} Sequence Variations")
print(f"Analysis: {analysis_description}")
print("=" * 60)

# Filter to only variations with DMS data for statistics
with_dms = with_effects.dropna(subset=['cell entry'])

print(f"Statistics based on {len(with_dms)} variations with DMS data:")
print(f"Mean effect: {with_dms['cell entry'].mean():.3f}")
print(f"Median effect: {with_dms['cell entry'].median():.3f}")
print(f"Std deviation: {with_dms['cell entry'].std():.3f}")
print(f"\nEffect range: {with_dms['cell entry'].min():.3f} to {with_dms['cell entry'].max():.3f}")

# Count by effect category
print("\nEffect Categories:")
print(f"  Beneficial (> 0.5): {(with_dms['cell entry'] > 0.5).sum()} ({(with_dms['cell entry'] > 0.5).sum() / len(with_dms) * 100:.1f}%)")
print(f"  Neutral (-0.5 to 0.5): {((with_dms['cell entry'] >= -0.5) & (with_dms['cell entry'] <= 0.5)).sum()} ({((with_dms['cell entry'] >= -0.5) & (with_dms['cell entry'] <= 0.5)).sum() / len(with_dms) * 100:.1f}%)")
print(f"  Deleterious (< -0.5): {(with_dms['cell entry'] < -0.5).sum()} ({(with_dms['cell entry'] < -0.5).sum() / len(with_dms) * 100:.1f}%)")

# Most tolerated variations
print("\nMost Tolerated Sequence Variations (highest cell entry):")
display(with_dms.nlargest(10, 'cell entry')[['site', 'mutation_type', 'cell entry', 'mutation_count', 'region']])

# Most deleterious variations
print("\nMost Deleterious Sequence Variations (lowest cell entry):")
display(with_dms.nsmallest(10, 'cell entry')[['site', 'mutation_type', 'cell entry', 'mutation_count', 'region']])

## Create interactive plots with Altair

In [ ]:
# Plot 1: Distribution histogram with all possible mutations and observed variations on separate y-axes
# Filter to only variations with DMS data for plotting
with_dms = with_effects.dropna(subset=['cell entry'])

# Define shared bin parameters with explicit step size to ensure both histograms have same bin widths
# Using a fixed step size guarantees identical bin boundaries for both datasets
bin_params = alt.Bin(step=0.2)

# Background histogram: All possible DMS mutations (blue) - right y-axis
hist_all_mutations = alt.Chart(cell_entry).mark_bar(color='blue', opacity=0.6).encode(
    alt.X('cell entry:Q', bin=bin_params, title='Cell Entry Effect'),
    alt.Y('count()', title='All possible mutations', axis=alt.Axis(titleColor='blue')),
    tooltip=['count()']
)

# Foreground histogram: Observed sequence variations (green) - left y-axis
hist_observed = alt.Chart(with_dms).mark_bar(color='green', opacity=0.7).encode(
    alt.X('cell entry:Q', bin=bin_params, title='Cell Entry Effect'),
    alt.Y('count()', title='Mutations in natural sequences', axis=alt.Axis(titleColor='green')),
    tooltip=['count()']
)

# Add zero reference line
rule_zero = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(color='red', strokeDash=[5, 5], size=2).encode(x='x:Q')

# Layer histograms with independent y-axes and resolve x scale to shared
chart1 = alt.layer(
    hist_all_mutations,
    hist_observed,
    rule_zero
).resolve_scale(
    y='independent',
    x='shared'
).properties(
    width=600,
    height=400,
    title={
        "text": f'{strain} {analysis_description}',
        "subtitle": f'Distribution of Cell Entry Effects | Blue: All Possible Mutations (n={len(cell_entry)}) | Green: Mutations in Natural Sequences (n={len(with_dms)})'
    }
)

chart1

In [ ]:
# Plot 2: Scatter plot by position
# Use only variations with DMS data
chart2 = alt.Chart(with_dms).mark_circle(opacity=0.7).encode(
    x=alt.X('site:Q', title='Site'),
    y=alt.Y('cell entry:Q', title='Cell Entry Effect'),
    size=alt.Size('mutation_count:Q', title='Number of sequences with mutation', scale=alt.Scale(range=[20, 400])),
    tooltip=['site', 'mutation_type', 'wildtype', 'mutant', 'mutation_count', 'cell entry']
).properties(
    width=700,
    height=400,
    title={
        "text": f'{strain}',
        "subtitle": f'{analysis_description} - Cell Entry Effects by Position'
    }
)

# Add zero line
zero_line = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(color='red', strokeDash=[5, 5]).encode(y='y:Q')

chart2 = chart2 + zero_line

chart2

## Save combined chart to HTML

In [ ]:
# Combine all charts vertically
combined_chart = alt.vconcat(
    chart1,
    chart2
).properties(
    title={
        "text": f'{strain} Sequence Variation Cell Entry Effects Analysis',
        "subtitle": f'{analysis_description} | Differences identified: {mutation_identified_relative_to} | Effects calculated: {effects_calculated_relative_to}'
    }
).configure_view(
    strokeWidth=0
).configure_axis(
    labelFontSize=12,
    titleFontSize=14
).configure_title(
    fontSize=20,
    anchor='middle',
    subtitleFontSize=14
).configure_legend(
    labelFontSize=12,
    titleFontSize=14,
    orient='right'
)

# Save to HTML
combined_chart.save(output_html)
print(f"Saved interactive plots to: {output_html}")

combined_chart